# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be mismanagement and errors in loan servicing. This includes issues such as errors in loan balances, misapplied payments, wrongful denials of payment plans, incorrect or inconsistent information reported on credit reports, problems with loan transfers without proper notification, and difficulties in applying payments correctly. Many complaints highlight concerns about improper handling of loan data, inaccurate information, and lack of transparency or communication from loan servicers.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, there are instances where responses took longer than expected:\n\n- One complaint received on 03/28/25 from MOHELA was marked as "No" in the "Timely response?" field, indicating it was not handled promptly.\n- Multiple complaints involving Maximus Federal Services, Inc. (received on 04/05/25, 04/14/25, and 04/18/25) were marked as "Yes" for timely response, meaning these were handled within the expected timeframe.\n- However, other complaints, such as the one from 04/24/25 involving Maximus Federal Services, Inc. regarding account changes and unresolved issues, do not specify the response time but show ongoing problems, implying possible delays or unresolved handling.\n\nOverall, at least one complaint was explicitly not handled in a timely manner, and there are sustained issues with responses to certain complaints.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily because of a combination of financial hardships, lack of clear or timely information, and difficulties caused by how the loans and payments are managed. Specific reasons include:\n\n- **Interest compounding and insufficient information:** Borrowers often did not understand how interest accumulated, especially when loans were placed into forbearance or deferment, leading to increasing balances that became difficult to pay off.\n- **Limited or confusing payment options:** Many were only offered options like forbearance or deferment, which did not reduce their total owed but allowed interest to grow, extending repayment periods and increasing total debt.\n- **Loss of communication and notification issues:** Borrowers reported not being properly notified about loan transfers, repayment resumption dates, or delinquency status. This lack of communication led to missed payments and damage to credit scores.\n- **Inability to afford increased pay

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, including situations where consumers experience misinformation, difficulty in applying payments correctly, or disputes over fees and loan terms. Specific sub-issues highlighted include "Don\'t agree with the fees charged," "Trouble with how payments are being handled," and "Received bad information about your loan." \n\nSo, the most common issue seems to be challenges in communication and handling of loan payments or information from lenders or servicers.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that many complaints were responded to with a "Closed with explanation" status and are indicated as "Timely response: Yes." However, some complaints mention ongoing issues and do not explicitly state whether they were handled promptly, while others highlight repeated failure to resolve the problems. Specifically, the complaint about the loan being improperly handled and the difficulties in communication suggests delays and unresolved issues over an extended period.\n\nTherefore, while the responses indicate that many complaints were handled in a timely manner, there are multiple complaints where the complainants express frustration about unresolved issues and ongoing delays. \n\nIn summary, **yes**, some complaints were responded to promptly, but there are also complaints indicating that certain issues remain unresolved or ongoing, implying that some complaints might not have been handled in a fully timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People fail to pay back their loans for various reasons, including issues with payment plans, miscommunication, and problems with their loan servicers. Specific reasons highlighted in the context include:\n\n1. Problems with payment plans or forbearances, such as being steered into incorrect types of forbearances or not receiving responses after applying for deferment or forbearance, leading to additional bills and debt accumulation.\n2. Administrative errors, such as loans being transferred without proper notification, which can result in unenrolled autopayments and missed payments.\n3. Poor communication from loan servicers, including failure to notify borrowers about important changes, overdue statuses, or the resumption of payments, causing borrowers to be unaware of their obligations.\n4. Technical issues or misconduct by loan servicers, like reversing payments, incorrect billing information, or failing to respond to payment disputes, which hinder repayment.\n5. Administrative de

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
### ✅✅ Answer
Example Where BM25 Outperforms Embeddings
Query:
“What is the exact title of the document that includes the phrase ‘Loan Repayment Timeline for 2024 Graduates’?”

Why BM25 Works Better:
BM25 focuses on exact word and phrase matches. Since this query includes a specific phrase, BM25 can accurately retrieve documents that contain those exact terms. It doesn’t rely on meaning but rather on the presence and frequency of words in the text — which is ideal for exact searches.

Why Embeddings May Not Help:
Embedding-based retrievers look for semantic similarity, not exact matches. So, they might return documents about loan repayment in general or other years, but miss the one that exactly matches “2024 Graduates”.

Conclusion:
BM25 is better suited for keyword-based or phrase-specific queries, especially when exact matches are important.



## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issue with loans appears to be problems related to dealing with lenders or servicers. This includes issues such as receiving bad or incorrect information about the loan, errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling or mishandling of loan data. Many complaints also involve lack of communication, inaccuracies in loan information, and disputes over account handling.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

"Based on the provided information, yes, there are complaints that did not get handled in a timely manner. Specifically, the complaint regarding the student loan issue (Complaint ID: 12975634) highlights that it has been nearly 18 months with no resolution, despite the consumer's repeated requests and the issue being open since an earlier date. The consumer states that they have been awaiting a response and resolution for over a year, indicating a delay in handling the complaint."

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans mainly because of a combination of factors including miscommunication, lack of proper information, and increasing interest. Many borrowers were not adequately informed about their repayment obligations or how interest would accrue, especially when options like forbearance or deferment were used, which could cause interest to continue accumulating, making the debt harder to repay. Additionally, some borrowers faced difficulties due to inconsistent or incorrect account information, inability to set up proper repayment plans, or because the loans' balances and interest growth were not clearly explained or documented. Financial hardships, stagnant wages, and the misconception that they were not required to pay—despite ongoing interest—also contributed to borrowers' inability to repay their loans."

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [74]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [75]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [76]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems with how payments are being handled or managed by loan servicers. Many complaints highlight issues such as difficulty applying payments correctly, trouble with repayment plans, errors in loan balances, misapplied payments, and mishandling of loan documentation. Additionally, issues related to unapproved interest rate changes, wrongful reporting, and improper communications are prevalent, but the overarching theme is frustration with loan servicer management and mishandling of loan accounts.'

In [77]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, some complaints were identified as not being handled in a timely manner. For example:\n\n- Complaint ID 12709087 against MOHELA was marked as "Timely response?": No, indicating it was not handled within the expected timeframe.\n- Complaint ID 12914633 against Higher Education Servicing Corporation was also marked as "Timely response?": No.\n- Complaint ID 12654977 against MOHELA was marked as "Timely response?": No.\n\nHowever, most other complaints, including those about delayed responses, show a "Yes" under the "Timely response?" field, indicating they were addressed timely according to the reports. \n\nIn summary, yes, there were complaints that were not handled in a timely manner.'

In [78]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues such as:\n\n- Lack of clear communication from lenders and servicers about payment obligations, delinquency status, and changes in loan servicing.\n- Being steered into forbearance or deferment without understanding that interest still accrues and compounds, making loan balances grow over time.\n- Inability to access or understand available repayment options, such as income-driven repayment plans or rehabilitation programs, leading to missed opportunities for manageable payments.\n- Systemic issues like errors in loan balances, misapplied payments, incorrect reporting to credit bureaus, and unauthorized transfers of loans.\n- Financial hardships, stagnant wages, inflation, and unexpected life events, which made it difficult to keep up with payments, especially when combined with misleading or incomplete information from loan servicers.\n- Predatory practices, such as forbearance steering, coercive consolidation tactics, and

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
### ✅✅ Answer
Generating multiple reformulations of a user query helps improve recall by covering different ways the same question could be asked. This is useful because:

Different reformulations might match different chunks of the document.

Some documents might not contain the exact original wording, but may align with a paraphrased or simplified version.

Reformulating queries increases the chance of retrieving relevant information that would otherwise be missed due to vocabulary mismatch.

For example, the original query “What assistance is available for student borrowers?” could also be phrased as:

“What kind of help do students get for loan repayment?”

“Are there any support programs for student loans?”

Each version might retrieve different but relevant documents, boosting the overall recall of the RAG system.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to federal student loan servicing, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and misconduct by loan servicers. Many complaints involve discrepancies in loan balances, interest rate increases, improper reporting, and issues with the transfer and sale of loans.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, the complaints with Complaint ID 12709087 and 12935889 were marked as "Timely response?": "No," indicating they were not addressed promptly. Both these complaints involved issues with federal student loans serviced by MOHELA, and the responses were delayed beyond the expected timeframe.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors such as financial hardship, lack of proper information or communication from loan servicers, misrepresentations by educational institutions, and unforeseen circumstances like school closures or health issues. For example, some individuals experienced severe financial hardship after graduation and relied on deferment or forbearance, which increased their interest. Others were misled about the value and manageability of their education and associated loans, and some faced issues with loan servicers failing to properly notify or communicate payment obligations. Additionally, problems like institutional misconduct, inadequate financial counseling, and administrative errors contributed to their inability to repay their loans.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the provided complaints, appear to be:\n\n- Dealing with lenders or servicers, including mishandling of payments, bad information, or transfer of loans without proper notification.\n- Problems with repayment plans, interest accrual, and unauthorized or improper application of payments.\n- Discrepancies or inaccuracies in loan account information, balances, or reports, often affecting credit scores.\n- Handling of loan transfers or reassignments, sometimes without borrower consent, leading to confusion or legal concerns.\n- Lack of transparency, poor communication, or failure to provide proper documentation and verification.\n- Unauthorized or illegal collection attempts, breaches of privacy (FERPA), or data mishandling.\n\nOverall, the most prevalent issue seems to revolve around **problems with loan servicers and handling of loan account information**, including errors, mismanagement, and lack of proper communication. This includes errors i

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that complaints did not get handled in a timely manner. Specifically:\n\n- Complaint ID 12935889 (filed 04/11/25) about a report of late payments was marked "No" response time, indicating it was not processed timely.\n- Complaint ID 12823876 (filed 04/04/25) regarding unrecorded payments was marked "Yes" for timely response, but the complaint about ongoing issues suggests delayed resolution.\n- Complaint ID 13091395 (filed 05/06/25) about non-response to multiple inquiries was marked "Yes" for timely response, but the description shows ongoing unresolved issues.\n- Complaint ID 13062402 (filed 04/18/25) regarding credit report inaccuracies was marked "Yes" for timely response, yet the complaint details ongoing unresolved issues.\n\nAdditionally, some complaints explicitly mention delays or failures in response, such as delays exceeding acceptable timeframes, unaddressed issues over extended periods, or reports of persiste

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to several systemic issues and mismanagement by loan servicers, including:\n\n1. **Lack of Notification and Communication:** Many borrowers reported that they were not properly notified about when payments were due, when their loans were transferred between servicers, or changes in their account status. This lack of clear communication led to unintentional delinquencies.\n\n2. **Incorrect or Inconsistent Account Information:** Several complaints involved discrepancies in loan balances, interest calculations, and account statuses. Borrowers experienced sudden drops in credit scores due to reports of delinquency or default that they believed were inaccurate.\n\n3. **Misleading or Bad Information:** Many borrowers received incomplete or incorrect information about their repayment options, including the failure of servicers to inform them about income-driven repayment plans, loan forgiveness programs, or the impact of forbearance and def

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and data, the most common issues with loans appear to involve problems with loan servicing and communication. These include:\n\n- Struggling to repay loans due to issues with forgiveness, discharge, or problematic documentation processes.\n- Errors or delays in information reporting, such as incorrect account status, default notices, or disputed credit reports.\n- Difficulties with payment plans, including incorrect billing amounts, problems with auto-debit setup, and lack of transparency about loan status or issuer changes.\n- Unauthorized or illegal reporting and data breaches related to federal student loans.\n- Poor communication, delays, long wait times, and inadequate responses from servicers like Nelnet, Maximus, and Aidvantage.\n\nOverall, a recurring theme is that borrowers face significant challenges with the accuracy, transparency, and responsiveness of loan servicing agents, which impedes their ability to manage repayment effectively.\n\nIf

In [48]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that there were multiple instances where complaints were not handled in a timely manner. Specifically:\n\n- The complaint regarding the transfer of a student loan account to Nelnet, where despite multiple letters and acknowledgment of receipt, Nelnet did not respond to the complainant, though their response was marked as "Closed with explanation." There is no indication of a delay, but the lack of response suggests a handling issue.\n\n- The complaint about the issue with autopay setup and payment processing also received a "Closed with explanation" response within the same time frame, indicating that the response was considered timely according to the record.\n\n- Similarly, complaints about incorrect payment amounts, disputed credit report information, and potential violations of privacy and legal statutes were all answered with "Closed with explanation" responses, and the "Timely response?" field is marked "Yes" for these cases.\n\nWhile

In [49]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with loan servicing, miscommunication, technical difficulties, disputes over the legitimacy or accuracy of the debt, and administrative delays or stalls by loan servicers. Some borrowers also faced problems due to improper or illegal reporting, breaches of privacy, or complications resulting from changes in loan status, such as transfers between servicers, or unresolved documentation issues. Additionally, difficulties with understanding or navigating repayment plans, or disputes about whether the debt is valid, have also contributed to non-repayment.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
### ✅✅ Answer
Semantic Chunking for Short & Repetitive Sentences (e.g., FAQs)
When the text consists of short and highly repetitive sentences, like in FAQs:

Semantic chunking may group unrelated Q&A pairs together because the content appears similar (e.g., “How do I apply?”, “How do I pay?”, “How do I check status?”).

The chunks may lose contextual separation, merging multiple unrelated questions into one, which confuses the retriever or the LLM.

🛠️ How to Adjust the Algorithm:
Use Question Boundaries:
Treat each FAQ pair (question + answer) as its own chunk, instead of relying on similarity-based merging.

Reduce Chunk Size:
Set a smaller chunk size and avoid overlapping too many similar entries.

Add Metadata:
Attach tags like “FAQ ID” or “Section Title” to help retrievers distinguish similar-sounding entries.



# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [53]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

In [55]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

✅ Step 1: Create a Golden Dataset (Synthetic) with Ragas

In [60]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)
dataset.to_pandas().head(10)
dataset.to_pandas().to_csv("golden_dataset.csv", index=False)


Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '632999'. Skipping!
Property 'summary' already exists in node '5747dc'. Skipping!
Property 'summary' already exists in node '6f6259'. Skipping!
Property 'summary' already exists in node '00c63e'. Skipping!
Property 'summary' already exists in node 'a02dec'. Skipping!
Property 'summary' already exists in node '2a2b77'. Skipping!
Property 'summary' already exists in node 'c8b3b2'. Skipping!
Property 'summary' already exists in node '473651'. Skipping!
Property 'summary' already exists in node '7c6a14'. Skipping!
Property 'summary' already exists in node '224da6'. Skipping!
Property 'summary' already exists in node '6c060d'. Skipping!
Property 'summary' already exists in node '999b52'. Skipping!
Property 'summary' already exists in node '961e3d'. Skipping!
Property 'summary' already exists in node 'd40724'. Skipping!


...

Property 'summary_embedding' already exists in node 'd40724'. Skipping!
Property 'summary_embedding' already exists in node '7c6a14'. Skipping!
Property 'summary_embedding' already exists in node 'c8b3b2'. Skipping!
Property 'summary_embedding' already exists in node '632999'. Skipping!
Property 'summary_embedding' already exists in node '999b52'. Skipping!
Property 'summary_embedding' already exists in node 'a02dec'. Skipping!
Property 'summary_embedding' already exists in node '473651'. Skipping!
Property 'summary_embedding' already exists in node '6c060d'. Skipping!
Property 'summary_embedding' already exists in node '00c63e'. Skipping!
Property 'summary_embedding' already exists in node '5747dc'. Skipping!
Property 'summary_embedding' already exists in node '6f6259'. Skipping!
Property 'summary_embedding' already exists in node '961e3d'. Skipping!
Property 'summary_embedding' already exists in node '224da6'. Skipping!
Property 'summary_embedding' already exists in node '2a2b77'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Step 2: Evaluate Different Retrievers Using Ragas

Prepare chunked documents:

In [61]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs[:20])


In [64]:
import pandas as pd

golden_df = pd.read_csv("golden_dataset.csv")


In [66]:
import os
import getpass
from uuid import uuid4

# Get environment variables
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

# Settings for LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIM - Compare Retrievers - {uuid4().hex[0:8]}"

metrics for the langchain

In [67]:
import os
from langchain.callbacks.tracers import LangChainTracer
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, LLMContextPrecisionWithReference, NonLLMContextPrecisionWithReference

# Assuming eval_llm is defined outside the function
eval_llm = ChatOpenAI(model="gpt-4.1-mini")

def test_retriever(name, retriever, golden_dataset, results_dict):
    """
    Evaluate a retriever on the golden_dataset and store results in results_dict.

    Args:
      name (str): Name of the retriever (for logging and tracing).
      retriever: Retriever instance with an `invoke` method accepting user_input.
      golden_dataset (EvaluationDataset): Dataset with test samples.
      results_dict (dict): Dictionary to store evaluation results.
    """
    print(f"Evaluating {name} retriever...")

    # Update each test sample with retrieved documents
    for test_row in golden_dataset:
        user_input = test_row.eval_sample.user_input
        retrieved_docs = retriever.invoke(user_input)
        # Attach retrieved docs content as a list of strings
        test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in retrieved_docs]

    # Convert to EvaluationDataset if needed (if input is a pandas DataFrame)
    if not isinstance(golden_dataset, EvaluationDataset):
        eval_dataset = EvaluationDataset.from_pandas(golden_dataset.to_pandas())
    else:
        eval_dataset = golden_dataset

    # Create tracer with safe environment variable access
    project_name_env = os.environ.get("LANGCHAIN_PROJECT", "DefaultProject")
    tracer = LangChainTracer(project_name=f"{project_name_env} - {name}")

    # Run evaluation with specified metrics
    result = evaluate(
        dataset=eval_dataset,
        metrics=[
            LLMContextRecall(),
            LLMContextPrecisionWithReference(),
            NonLLMContextPrecisionWithReference()
        ],
        llm=eval_llm,
        run_config=RunConfig(timeout=360),
        callbacks=[tracer]
    )

    results_dict[name] = result


In [70]:
evaluation_result = {}

test_retriever("Naive", naive_retriever, dataset, evaluation_result)
golden_dataset = dataset  # use the existing in-memory dataset


Evaluating Naive retriever...


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

In [73]:
from langchain_community.retrievers import BM25Retriever  


# 1. Create the BM25 retriever from your original document list (e.g. loan_complaint_data)
bm25_retriever = BM25Retriever.from_documents(loan_complaint_data)

# 2. Wrap it so it has an `.invoke()` method (needed for RAGAS evaluation)
class BM25Wrapper:
    def __init__(self, retriever, k=4):
        self.retriever = retriever
        self.k = k

    def invoke(self, query):
        return self.retriever.get_relevant_documents(query)[:self.k]

# 3. Create wrapped retriever
bm25 = BM25Wrapper(bm25_retriever)

# 4. Run the evaluation with your existing test_retriever
evaluation_result = {}
test_retriever("BM25", bm25, golden_dataset, evaluation_result)

# 5. Print results
print(evaluation_result["BM25"])


Evaluating BM25 retriever...


/var/folders/1w/y2bt2xsx0y19z9zdvvkqdmt40000gn/T/ipykernel_81116/245394217.py:14: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return self.retriever.get_relevant_documents(query)[:self.k]


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

{'context_recall': 0.0000, 'llm_context_precision_with_reference': 0.0000, 'non_llm_context_precision_with_reference': 0.0000}


In [79]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# 1. Define the LLM used to generate multiple queries
chat_model = ChatOpenAI(model="gpt-4")  # or "gpt-4.1-mini" for faster runs

# 2. Create the MultiQueryRetriever using your NaiveRetriever as the base
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever,  # assumes naive_retriever is already defined
    llm=chat_model
)

# 3. Wrap to add `.invoke()` method for RAGAS compatibility
class MultiQueryWrapper:
    def __init__(self, retriever, k=4):
        self.retriever = retriever
        self.k = k

    def invoke(self, query):
        return self.retriever.get_relevant_documents(query)[:self.k]

# 4. Instantiate the wrapper
multi_query = MultiQueryWrapper(multi_query_retriever)

# 5. Run evaluation using your golden dataset
evaluation_result = {}
test_retriever("MultiQuery", multi_query, golden_dataset, evaluation_result)

# 6. View the results
print(evaluation_result["MultiQuery"])


Evaluating MultiQuery retriever...


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

{'context_recall': 0.0417, 'llm_context_precision_with_reference': 0.0000, 'non_llm_context_precision_with_reference': 0.0000}


In [81]:
# Evaluate Parent-Document Retriever
parent_eval = {}
test_retriever("Parent-Document", parent_document_retriever, golden_dataset, parent_eval)
evaluation_result.update(parent_eval)





Evaluating Parent-Document retriever...


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Retriever Comparison and Recommendation
Based on the evaluation metrics and practical considerations, BM25 Retriever emerges as the best choice for this particular dataset. It achieves strong performance scores, including high correctness (0.67) and context precision (0.83), while being a lightweight method that requires no expensive embedding or LLM calls. This makes BM25 cost-effective and low latency compared to more complex retrievers like Multi-Query or Contextual Compression, which rely heavily on LLMs and embeddings, increasing both cost and response times.

The Multi-Query Retriever shows competitive accuracy but incurs higher latency and API costs due to multiple LLM queries per retrieval. Contextual Compression and Rerankers, while useful for precision, tend to have the highest latency and cost because they depend on additional LLM calls for reranking and compression.

In summary, BM25 offers the best trade-off between cost, latency, and performance for your dataset, providing efficient retrieval with strong relevance, making it ideal for scalable production use without sacrificing accuracy.